In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from training.feature_engineering import load_xy

# 03 - Baseline models
Logistic regression baseline on a 150k sample; then the full-data comparison table written by `training/compare_models.py`.

In [2]:
Xtr, ytr = load_xy("train", "delay", sample=0.04); Xva, yva = load_xy("valid", "delay", sample=0.2)
from sklearn.metrics import roc_auc_score
lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=300)).fit(Xtr, ytr)
print("val AUC (sample baseline):", round(roc_auc_score(yva, lr.predict_proba(Xva)[:, 1]), 4))
coef = pd.Series(lr[-1].coef_[0], index=Xtr.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 8)); coef.plot.barh(ax=ax, color=np.where(coef > 0, "#dc2626", "#16a34a")); ax.set_title("logistic regression coefficients (standardised)")
save("logreg_coefficients")

val AUC (sample baseline): 0.6673


In [3]:
M = json.loads((C.METRICS_DIR / "model_metrics.json").read_text())
rows = [{"model": m["model_name"], "target": m["target"], **{k: m["metrics"].get(k) for k in ("accuracy","precision","recall","f1","roc_auc","brier")}}
        for m in M["models"] if m["task"] == "classifier"]
pd.DataFrame(rows).round(4)

,model,target,accuracy,precision,recall,f1,roc_auc,brier
0,logreg,delay,0.6628,0.2275,0.5221,0.3169,0.6460,0.2002
1,random_forest,delay,0.6777,0.2370,0.5186,0.3253,0.6568,0.1881
2,xgboost,delay,0.6747,0.2384,0.5336,0.3296,0.6611,0.1244
3,neural_net,delay,0.6543,0.2274,0.5451,0.3209,0.6511,0.1984
4,xgboost,cancel,0.9910,0.0444,0.0960,0.0608,0.7061,0.0031
